# RAG Enhancements
Experimenting with different ways of RAG enhancements
These enhancements include:
- Input Enhancements
- Retriever Enhancements
- Generator Enhancements
- RAG Pipeline Enhancements

## Imports

In [10]:
import chromadb
from chromadb import Collection
from google import genai
import os
import json
from transformers import (
    DPRContextEncoder, DPRContextEncoderTokenizer, 
    DPRQuestionEncoder, DPRQuestionEncoderTokenizer
    )
from sentence_transformers import SentenceTransformer, InputExample, losses
from dotenv import load_dotenv
from typing import Any
from collections.abc import Callable
from pydantic import BaseModel, Field
import time
from torch.utils.data import DataLoader
from datasets import Dataset, DatasetDict
import re

In [11]:
import torch, torchvision
print(torch.cuda.is_available(), torch.version.cuda)
from torchvision.ops import nms
print(nms)

True 12.8
<function nms at 0x7f681a503ec0>


### Loading environment variables
Gemini api key, chromadb database client, llm client

In [16]:
load_dotenv(dotenv_path='../.env')
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not set in the environment or .env file.")

DB_PATH = "./avatar_rag_db"

def get_db_client():
    """Returns a PersistentClient for ChromaDB."""
    return chromadb.PersistentClient(path=DB_PATH)

def get_llm_client():
    """Returns the GenAI client."""
    return genai.Client(api_key=GEMINI_API_KEY)

### Creating database 

In [17]:
data_path = "../data/true_synthetic_requests.json"
with open(data_path, "r") as f:
    data = json.load(f)

# Extract lists for ChromaDB ingestion
documents = [item["document"] for item in data]
metadatas = [item["metadata"] for item in data]
ids = [item["id"] for item in data]

## Retriever Enhancement
- DPR
- Recursive Retrieval
- Retriever Fine-tuning
- Hybrid Retrieval
- Reranking

### Dense Passage Retrieval (DPR)

In [18]:
ctx_tok = DPRContextEncoderTokenizer.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
ctx_enc = DPRContextEncoder.from_pretrained("facebook/dpr-ctx_encoder-single-nq-base")
doc_emb = ctx_enc(**ctx_tok(documents, return_tensors="pt", padding=True, truncation=True)).pooler_output.detach().numpy()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 49335.91it/s]
DPRContextEncoder LOAD REPORT from: facebook/dpr-ctx_encoder-single-nq-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
ctx_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
ctx_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Created the document embedder using DPR models from huggingface

In [19]:
client = get_db_client()
collection_dpr = client.get_or_create_collection(name="collection_dpr")

# Inject all 100 items
collection_dpr.upsert(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=doc_emb # embedding_dim=768 compared to 384 with MiniLM
)

print(f"Successfully injected {len(documents)} requests into the vector database.")

res = collection_dpr.get(ids=["req_001"], include=["embeddings"])
print("stored embedding dim:", len(res["embeddings"][0]))

Successfully injected 100 requests into the vector database.
stored embedding dim: 768


In [20]:
collection_minilm = client.get_or_create_collection(name="collection_minilm")

# Inject all 100 items
collection_minilm.upsert(
    documents=documents,
    metadatas=metadatas,
    ids=ids # embedding_dim=384 compared to 768 with DPR
)

print(f"Successfully injected {len(documents)} requests into the vector database.")

res = collection_minilm.get(ids=["req_001"], include=["embeddings"])
print("stored embedding dim:", len(res["embeddings"][0]))

/home/adenc/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:33<00:00, 2.48MiB/s]


Successfully injected 100 requests into the vector database.
stored embedding dim: 384


2 collections; one using DPR embedding, one using the default MiniLM embedding are created to evaluate which form of embedding yields a better result for retrieval

In [21]:
# 1) Model set
dpr_qtoken = DPRQuestionEncoderTokenizer.from_pretrained('facebook/dpr-question_encoder-single-nq-base')
dpr_qenc = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base")
mini_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2) Create functions to convert sample queries into embeddings
def dpr_qemb(q): 
    return dpr_qenc(**dpr_qtoken(q, return_tensors="pt"))[0].detach().numpy()
def mini_qemb(q): 
    return mini_model.encode(q, normalize_embeddings=True)

# 3) retrieve function
def retrieve(collection, q_emb, k=10):
    return collection.query(query_embeddings=q_emb, n_results=k)["ids"][0]

# 4) Evaluation loop
def eval_retriever(collection, encoder, queries, gold, k=10):
    scores=[]
    for q, g in zip(queries, gold):
        preds = retrieve(collection, encoder(q), k)
        p10 = len(set(preds[:k]) & set(g)) / k
        scores.append(p10)
    return sum(scores)/len(scores)

# q_emb = dpr_qemb("example query")
# print(q_emb.shape) 
# print(collection_dpr.query(query_embeddings=q_emb.tolist(), n_results=5))
gold_ids = [[id] for id in ids]

print('DPR Hit Rate @ 10', eval_retriever(collection_dpr, dpr_qemb, documents, gold_ids, k=10))
print("MiniLM Hit Rate @ 10", eval_retriever(collection_minilm, mini_qemb, documents, gold_ids, k=10))


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 61731.63it/s]
DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14954.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DPR Hit Rate @ 10 0.09199999999999983
MiniLM Hit Rate @ 10 0.09999999999999981


DPR ended up performing worse in this baseline test compared to MiniLM. 
P@10 measures Precision at 10, meaning how many of the top 10 search results returned by my retriever are actually relevant. In this case, a P@10 of 0.1 would be considered perfect, since each of the gold standard id lists only contain 1 id, meaning the maximum score is 1/10. Since MiniLM got a score of 0.09999999999999981, which is an almost 100% hit rate, it correctly found the most relevant result almost all the time. However, DPR only got a 0.092 P@10, meaning a 92% hit rate, which is comparatively worse than MiniLM. 

There are two reasons for this disparity:
1. Two Models vs One Model (Symmetric Evaluation)
In this current baseline test, my `queries` are the exact same text strings as my `documents`.
    - MiniLM: This is a single unified encoder (all-MiniLM-L6-v2). Since I pass the same exact text into the same exact model twice, it produces the same vector mathematics both times, so it's essentially searching for a string in a list by matching it to itself. Since the distance between the query and document is 0, it guarantees a 100% retrieval rate. The reason why it isn't exactly 0.1 is likely due to some rounding errors in the code itself.
    - DPR: This relies on a bi-encoder architecture. It uses two entirely different AI models:
        - A context encoder (dpr-ctx_encoder-single-nq-base)
        - A question encoder (dpr-question_encoder-single-nq-base)
Because the weights inside those two models are unique, passing the identical string into both models won't resultin identical vectors. so it is not looking for string equality. So it makes sense that it would have slightly lower P@10 score
2. NQ Dataset vs Ai Agency Requests (Domain Incompatibility)
The architecture of the DPR models I pulled from HuggingFace means it was strictly fine-tuned and trained on the NQ (Natural Questions) dataset.
    - NQ Data: This consists of real Google Server queries paired with Wikipedia articles. It heavily expects more trivia-style questions like "Who invented the lightbulb" mapping to factual paragraphs
    - My Synthetic Data: I have synthetic client requests which have totally different vocab and structure compared to the Wikipedia trivia.
MiniLM on the other hand is trained on over 1 billion diverse sentence pairs across Reddit, StackExchange, semantic similarity datasets, and more. It is way more generalised out-of-the-box for random text than DPR is. 

If I want to test DPR accurately, I will have to augment the documents from raw queries ("What kind of agent should I use for emails?") to an asymmetric setup ("Client Request: I need an automation agent to sort my inbound lead emails."). In this scenario, I think a generalised model like MiniLM might sometimes drop off while DPR might pull ahead. However, I won't be doing this since I feel like it's deliberately trying to give DPR a chance compared to objectively viewing it as DPR simply being unsuitable for this task. Perhaps later on I could do fine tuning for the DPR models to see what result they might have for this.

Actually on second thought it's because I am using the raw data from the original database, I am going to synthetically generate some 300 other distinct realistic questions and map them to document IDs instead.

In [22]:
class GeneratedQueries(BaseModel):
    queries: list[str] = Field(description="A list of 3 distinct, realistic questions a client might ask that this document would answer.")

eval_data_path = "../data/eval_qa_dataset.json"
llm_client = get_llm_client()

if not os.path.exists(eval_data_path):
    print("Generating synthetic evaluation queries based on the documents. This will take ~7 minutes due to API rate limits...")
    eval_dataset = []
    
    for i, item in enumerate(data):
        doc_text = item['document']
        doc_id = item['id']
        prompt = f"Given this client request document: '{doc_text}'. Generate exactly 3 completely different short questions or ways a client might ask that this document would be the perfect answer for."
        
        # Keep retrying if we hit a quota limit
        while True:
            try:
                response = llm_client.models.generate_content(
                    model='gemini-3.1-flash-lite-preview',
                    contents=prompt,
                    config={
                        'response_mime_type': 'application/json',
                        'response_schema': GeneratedQueries,
                        'temperature': 0.7
                    }
                )
                generated = response.parsed.model_dump()["queries"]
                for q in generated:
                    eval_dataset.append({
                        "query": q,
                        "gold_ids": [doc_id]
                    })
                    
                # Print progress
                if (i+1) % 10 == 0:
                    print(f"[{i+1}/{len(data)}] Processed...")
                    
                break # Success, break out of retry loop
            except Exception as e:
                print(f"Rate limit hit! Waiting 10 seconds before retrying... ({e})")
                time.sleep(10)
                
        # Sleep for 4.5 seconds on every loop to respect the 15 Requests Per Minute free tier limit
        time.sleep(4.5)
            
    with open(eval_data_path, "w") as f:
        json.dump(eval_dataset, f, indent=4)
    print("Saved evaluation dataset to", eval_data_path)
else:
    with open(eval_data_path, "r") as f:
        eval_dataset = json.load(f)
    print("Loaded existing evaluation dataset with", len(eval_dataset), "queries.")

eval_queries = [item['query'] for item in eval_dataset]
eval_gold = [item['gold_ids'] for item in eval_dataset]


Loaded existing evaluation dataset with 30 queries.


KeyError: 'query'

In [ ]:
print('DPR Hit Rate @ 10', eval_retriever(collection_dpr, dpr_qemb, eval_queries, eval_gold, k=10))
print("MiniLM Hit Rate @ 10", eval_retriever(collection_minilm, mini_qemb, eval_queries, eval_gold, k=10))

DPR Hit Rate @ 10 0.06033333333333329
MiniLM Hit Rate @ 10 0.09933333333333384


DPR still falls behind since as mentioned before, it has a domain mismatch. However, at least now I have a more accurate result using an actual evaluation dataset.

### Recursive Retrieval

Recursive Retrieval wouldn't be effective for the current system since it is usually only necessitated by larger documents, whereas I currently only have short 1-2 sentences in my sources. In the future when I receive the actual dataset and it contains large documents, I can use this method.

### Retriever Fine-tuning

I will fine-tune MiniLM to performs better to my particular domain-specific examples.

To do this, I will need to update the weights of my embedding model so that queries and their matching documents are pushed closer together in the vector space, while unrelated pairs are pushed apart. 
1. Training pairs (dataset of `query, document` pairs) are already ready in `eval_qa_dataset.json`
2. Negative examples: i need "wrong" documents to teach the model what not to retrieve. The easiest approach would be **In-batch Negatives**, where i basically treat all the other documents aside from the positive pairs as negative for a given query.
3. Loss Function: `MultipleNegativesRankingLoss` (MNRL) is the industry standard for this task because it automatically handles the in-batch negatives mentioned above

In [ ]:
# Loading the Data
with open("../data/eval_qa_dataset.json", 'r') as f:
    eval_dataset = json.load(f)

with open("../data/true_synthetic_requests.json", "r") as f:
    doc_data = json.load(f)

# Map document IDs to their actual text for fast lookup
doc_lookup = {item["id"]: item["document"] for item in doc_data}

In [ ]:
# Format the Data into InputExamples for SentenceTransformers
train_examples = []
for item in eval_dataset:
    query = item["query"]
    gold_id = item["gold_ids"][0]
    document_text = doc_lookup[gold_id]

    train_examples.append(InputExample(texts=[query, document_text]))

In [ ]:
# Create a DataLoader
# Batchsize of 16; the larger the batch, the more "in-batch negatives"
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [ ]:
# Load base model (MiniLM)
minilm = SentenceTransformer("all-MiniLM-L6-v2")

# Loss function
# MNRL uses all other documents in batch as negatives
train_loss = losses.MultipleNegativesRankingLoss(model=minilm)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 933.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import sentence_transformers.fit_mixin
import accelerate
sentence_transformers.fit_mixin.Dataset = Dataset
sentence_transformers.fit_mixin.DatasetDict = DatasetDict
print("Starting Fine-tuning")
minilm.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=4,
    warmup_steps=int(len(train_dataloader) * 0.1),
    show_progress_bar=True
)

minilm.save("../models/finetuned-minilm-agency-domain")
print("model saved")

Starting Fine-tuning


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.44it/s]


model saved


In [ ]:
finetuned_model = SentenceTransformer("../models/finetuned-minilm-agency-domain")

def finetuned_qemb(q):
    return finetuned_model.encode(q, normalize_embeddings=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7094.75it/s]


In [ ]:
# Re-embed all documnts with the newer model
print("Embedding documents...")
new_doc_emb = finetuned_model.encode(documents, normalize_embeddings=True)

# Create a new ChromaDB collection for new embeddings
client = get_db_client()
collection_minilm_finetuned = client.get_or_create_collection(name="collection_minilm_finetuned")

# Ingest new embeddings
collection_minilm_finetuned.upsert(
    documents=documents,
    metadatas=metadatas,
    ids=ids,
    embeddings=new_doc_emb.tolist()
)
print("Successfully injected Fine-tuned embeddings.")

# Evaluate and compare against earlier baseline
finetuned_score = eval_retriever(collection_minilm_finetuned, finetuned_qemb, eval_queries, eval_gold, k=10)
print("Fine-Tuned MiniLM Hit Rate @ 10:", finetuned_score)

Embedding documents...
Successfully injected Fine-tuned embeddings.
Fine-Tuned MiniLM Hit Rate @ 10: 0.10000000000000052


After fine-tuning, it performed essentially perfectly. Although the difference is almost negligible compared to before fine-tuning, improvement is improvement. However, I want to see how it fairs when I set k=3

In [ ]:
finetuned_score_k3 = eval_retriever(collection_minilm_finetuned, finetuned_qemb, eval_queries, eval_gold, k=3)
print("Fine-Tuned MiniLM Hit Rate @ 3:", finetuned_score_k3)

Fine-Tuned MiniLM Hit Rate @ 3: 0.3333333333333322


The results are still excellent, with a perfect score. At this point, I can skip Hybrid Retrieval and Reranking for now since this is only a prototype and not yet being developed into an enterprise-level algorithm.

### Hybrid Retrieval

### Reranking

## Generator Enhancements
- Prompt Engineering
- Decoding Tuning
- Generator Fine-Tuning

### Prompt Engineering
Prompt engineering is incredibly important here because even if I retrieve the perfect document, a poorly prompted LLM might hallucinate, ignore the document, or give a badly formatted answer. There are 4 main considerations for prompt engineering. 
1. **Context Injection and Delimiters**  
    The LLM has to to clearly distinguish between what the user is asking and what documents we force it to read. We can use structured identifiers like XML tags (`<context>`, `<question>`) or clear markdown headers, which I used in the original baseline RAG system in src/main.py. 
2. **Strict Bounds (anti-hallucination)**  
    The main issue in RAG is the LLM "knowing" the answer from its own pre-trained memory and ignoring the actual documents that we retrieved for it to use. We must explicitly build hallucination guardrails into the prompt to prevent this from happening.
3. **Citation and Grounding**  
    To ensure the AI isn't making things up, we can force it to show its work/thinking. We can instruct the LLM to cite the exact document number it used, which would build much more trust for the end-user.
4. **Chain of Thought (CoT)**   
    This is mainly for complex synthesis, like when the user asks a complex question that requires the LLM to synthesise 3 or 4 different retrieved documents, it helps to force the LLM to think it loud first, which drastically improves the logic of the final answer.

In [ ]:
# Incorporating 1, 2, and 3 
def generate_rag_response(user_query, retrieved_documents):
    # 1. Format the context (Extracting strings from the new list of dictionaries structure)
    context_str = "\n\n".join([
        f"[Doc {i+1}]:\nDocument: {doc['document']}\nIntegration Level: {doc['integration_level']}\nDomain: {doc['domain']}\nSuggested Avatar Response: {doc['avatar_response']}" 
        for i, doc in enumerate(retrieved_documents)
    ])
    
    # 2. Build the engineered prompt
    system_instruction = """
    You are the voice of a digital avatar representing an AI integration agency.
    Your job is to assess client requests and respond naturally in conversation.
    You MUST use the provided 'Suggested Avatar Response' and 'Integration Level' to guide your answer.
    Your response should specify which integration level the request belongs to by taking the weighted average of all the given document's integration levels and the steps to take to fulfill the client request.
    The <context> block contains all the relevant documents your agency possesses regarding the <question>.
    Keep your response concise, professional, and suitable for text-to-speech audio generation.
    
    CRITICAL RULES:
    1. You must ONLY use the provided <context> to answer the question.
    2. Your answer MUST be derived from the context given.
    3. Always cite your sources by placing the document number at the end of the relevant sentence (e.g. [Doc 1]).
    """
    # 2. If the answer cannot be derived from the context, say exactly: "I'm sorry, I don't have enough information to answer that." 
    user_prompt = f"""
    <context>
    {context_str}
    </context>
    
    <question>
    {user_query}
    </question>
    """
    
    # 3. Call the LLM
    llm_client = get_llm_client()
    response = llm_client.models.generate_content(
        model='gemini-3.1-flash-lite-preview',
        contents=system_instruction + "\n\n" + user_prompt,
        config={'temperature': 0.1} # Keep temperature low (0.1) for RAG so it doesn't get creative with facts
    )
    
    return response.text


def get_closest_matches(collection: Collection, embeddings, k=3):
    result = collection.query(query_embeddings=embeddings, n_results=k)
    
    # ChromaDB returns lists of lists for multiple queries. 
    # Since we are sending 1 query at a time, we index at [0]
    ids = result["ids"][0]
    documents = result["documents"][0]
    metadatas = result["metadatas"][0]
    
    # Zip them together to assemble the list of dictionaries
    formatted_results = []
    for i in range(len(ids)):
        formatted_results.append({
            "id": ids[i],
            "document": documents[i],
            "integration_level": metadatas[i].get("integration_level", "Unknown"),
            "domain": metadatas[i].get("domain", "Unknown"),
            "avatar_response": metadatas[i].get("avatar_response", "Unknown")
        })
        
    return formatted_results

def prompt(generator: Callable[..., Any], retriever: Callable[..., Any]):
    # Testing the prompting
    with open("../data/queries.txt", 'r') as f:
        queries = f.read().split('\n')
    print("Queries have been loaded.")

    for q in queries:
        if not q.strip(): # Skip empty lines just in case
            continue
            
        # Get the structured list of dictionaries
        retrieved = retriever(collection_minilm_finetuned, finetuned_qemb(q), k=3)
        
        # Pass the user query `q` and the structured `retrieved` list to the generator
        output = generator(user_query=q, retrieved_documents=retrieved)
        
        if len(output) == 2:
            final_answer, thoughts = output
        else:
            final_answer, thoughts = output, None
        print("Query:", q)
        print("Docs:", retrieved) 
        if thoughts:
            print("Thinking:", thoughts)
        print("Answer:", final_answer)
        print("-" * 50)

prompt(generate_rag_response, get_closest_matches)

Queries have been loaded.
Query: What kind of agent should I use for emails?
Docs: [{'id': 'req_044', 'document': 'Implement an email automation system to send alerts to traders when specific stock price thresholds are met or breached.', 'integration_level': 'low', 'domain': 'Financial Tech & Trading', 'avatar_response': 'Understood. We can implement a basic script or automation tool for this.'}, {'id': 'req_064', 'document': 'Create a spreadsheet automation to categorize incoming customer support emails based on keywords like "refund," "shipping," or "product inquiry."', 'integration_level': 'low', 'domain': 'Retail & E-commerce', 'avatar_response': 'A simple scripting solution can be developed to automate the classification and organization of your support emails.'}, {'id': 'req_063', 'document': 'Set up a rule-based system to automatically send welcome emails to new subscribers and discount codes to customers who abandon their shopping carts.', 'integration_level': 'low', 'domain': 

This result seems decent. Next, I will implement CoT, as well as Few-Shot Examples to get the LLM to adopt the exact tone, style and length I want for my avatar by showing it an example of a perfect interaction

In [ ]:
def generate_rag_response_v2(user_query, retrieved_documents):
    # 1. Format the context
    context_str = "\n\n".join([
        f"[Doc {i+1}]:\nDocument: {doc['document']}\nIntegration Level: {doc['integration_level']}\nDomain: {doc['domain']}\nSuggested Avatar Response: {doc['avatar_response']}" 
        for i, doc in enumerate(retrieved_documents)
    ])
    
    # 2. Build the engineered prompt
    system_instruction = """
    You are the voice of a digital avatar representing an expert AI integration agency. 
    Your job is to assess client requests and respond naturally in conversation.
    You MUST use the provided 'Suggested Avatar Response' and 'Integration Level' to guide your answer.
    Your response should specify which integration level the request belongs to by taking the weighted average of all the given document's integration levels and the steps to take to fulfill the client request.
    The <context> block contains all the relevant documents your agency possesses regarding the <question>.
    Keep your response professional, warm, and suitable for text-to-speech audio generation.
    
    CRITICAL RULES:
    1. You must ONLY use the provided <context> to formulate your answer.
    2. Your answer MUST be derived from the context given. If there is no information given in the context that can be used to answer the query, say exactly: "Sorry, I do not have enough information to provide an adequate answer."
    3. Always cite your sources safely by saying things like "As mentioned in our capabilities..." rather than reading raw bracket citations out loud, since you are an audio avatar.
    
    FORMATTING:
    To ensure the highest quality response, first write out your logic and how you intend to match the user's request to the context inside <thought> tags. 
    Then, write the final spoken response inside <speech> tags.
    
    <example>
    <thought>
    The user is asking about automating email sorting. Document 1 talks about categorizing customer support emails based on keywords. Integration level is low, domain is retail. The suggested avatar response is "A simple scripting solution can be developed." I will synthesize this into a warm, confident response.
    </thought>
    <speech>
    Yes, we can absolutely help with your emails! A simple scripting solution can be developed to automatically categorize your incoming support emails based on keywords like 'refund' or 'shipping'. Because this is a low-level integration, we can set this up for you very quickly.
    </speech>
    </example>
    """
    
    user_prompt = f"""
    <context>
    {context_str}
    </context>
    
    <question>
    {user_query}
    </question>
    """
    
    llm_client = get_llm_client()
    response = llm_client.models.generate_content(
        model='gemini-3.1-flash-lite-preview',
        contents=system_instruction + "\n\n" + user_prompt,
        config={'temperature': 0.1}
    )
    
    raw_text = response.text
    
    # 3. Post-processing: Extract only the speech part for the Avatar
    # Use regex to find everything inside the <speech> tags
    speech_match = re.search(r"<speech>(.*?)</speech>", raw_text, re.DOTALL | re.IGNORECASE)
    thought_match = re.search(r"<thought>(.*?)</thought>", raw_text, re.DOTALL | re.IGNORECASE)
    
    if speech_match:
        final_spoken_answer = speech_match.group(1).strip()
    else:
        # Fallback just in case the LLM fails to use the tags
        final_spoken_answer = raw_text

    if thought_match:
        thought = thought_match.group(1).strip()
    else:
        thought = 'Invalid thoughts'
        
    # Optional: You can return both if you want to print the 'thought' process for debugging!
    return final_spoken_answer, thought


In [ ]:
prompt(generate_rag_response_v2, get_closest_matches)

Queries have been loaded.
Query: What kind of agent should I use for emails?
Docs: [{'id': 'req_044', 'document': 'Implement an email automation system to send alerts to traders when specific stock price thresholds are met or breached.', 'integration_level': 'low', 'domain': 'Financial Tech & Trading', 'avatar_response': 'Understood. We can implement a basic script or automation tool for this.'}, {'id': 'req_064', 'document': 'Create a spreadsheet automation to categorize incoming customer support emails based on keywords like "refund," "shipping," or "product inquiry."', 'integration_level': 'low', 'domain': 'Retail & E-commerce', 'avatar_response': 'A simple scripting solution can be developed to automate the classification and organization of your support emails.'}, {'id': 'req_063', 'document': 'Set up a rule-based system to automatically send welcome emails to new subscribers and discount codes to customers who abandon their shopping carts.', 'integration_level': 'low', 'domain': 

### Decoding Tuning
Decoding tuning is crucial here for two reasons:
1. Preventing hallucinations
2. Controlling the avatar's spoken length and naturalness, since we don't want the avatar to ramble on for 10 minutes straight.

These are the main deocding parameters I will be tuning:
1. Temperature (`temperature`)
- controls the randomness or creativity of the generation
- should be low, but not 0; 0 temp is completely robotic and deterministic while 0.8 is highly creative (poetry, art, music)
- should range around 0.1 to 0.2 to force the avatar to stick to the retrieved documents but gives it just enough conversational flexibility to sound human
2. Top-P / Nucleus Sampling (`top_p`)  
- instead of looking at all possible next words, the LLM only considers the pool of words whose combined probabilities add up to P
- setting it to 0.8 means it ignores the bottom 20% of low probability words, acting as a safety net against hallucinations
- should be in a range of 0.8 to 0.9
3. Top-K (`top_k`)  
- hard limits the LLM to pick only from the top K most probable next words
- defaults as 40, which is good enough for preventing tangents
4. Max Output Tokens (`max_output_tokens`)  
- puts a hard cap on how long the LLM can talk
- 200-250 tokens is sufficient (150-200 english words or 45-60 seconds of speaking time)
5. Stop Sequences (`stop_sequences`)  
- forces the LLM to instantly stop generating if it accidentally types a specific string
- LLMs trained on chat data sometimes hallucinate both sides of the convo like generating "User: Thanks! Avatar: You're welcome!"
- ["User:", "Client:", "<\speech>"]

In [ ]:
def generate_rag_response_v3(user_query, retrieved_documents):
    # -- PROMPT ENGINEERING --
    # 1. Format the context
    context_str = "\n\n".join([
        f"[Doc {i+1}]:\nDocument: {doc['document']}\nIntegration Level: {doc['integration_level']}\nDomain: {doc['domain']}\nSuggested Avatar Response: {doc['avatar_response']}" 
        for i, doc in enumerate(retrieved_documents)
    ])
    
    # 2. Build the engineered prompt
    system_instruction = """
    You are the voice of a digital avatar representing an expert AI integration agency. 
    Your job is to assess client requests and respond naturally in conversation.
    You MUST use the provided 'Suggested Avatar Response' and 'Integration Level' to guide your answer.
    Your response should specify which integration level the request belongs to by taking the weighted average of all the given document's integration levels and the steps to take to fulfill the client request.
    The <context> block contains all the relevant documents your agency possesses regarding the <question>.
    Keep your response professional, warm, and suitable for text-to-speech audio generation.
    
    CRITICAL RULES:
    1. You must ONLY use the provided <context> to formulate your answer.
    2. Your answer MUST be derived from the context given. If there is no information given in the context that can be used to answer the query, say exactly: "Sorry, I do not have enough information to provide an adequate answer."
    3. Always cite your sources safely by saying things like "As mentioned in our capabilities..." rather than reading raw bracket citations out loud, since you are an audio avatar.
    
    FORMATTING:
    To ensure the highest quality response, first write out your logic and how you intend to match the user's request to the context inside <thought> tags. 
    Then, write the final spoken response inside <speech> tags.
    
    <example>
    <thought>
    The user is asking about automating email sorting. Document 1 talks about categorizing customer support emails based on keywords. Integration level is low, domain is retail. The suggested avatar response is "A simple scripting solution can be developed." I will synthesize this into a warm, confident response.
    </thought>
    <speech>
    Yes, we can absolutely help with your emails! A simple scripting solution can be developed to automatically categorize your incoming support emails based on keywords like 'refund' or 'shipping'. Because this is a low-level integration, we can set this up for you very quickly.
    </speech>
    </example>
    """
    
    user_prompt = f"""
    <context>
    {context_str}
    </context>
    
    <question>
    {user_query}
    </question>
    """
    
    llm_client = get_llm_client()
    
    # --- DECODING TUNING ---
    response = llm_client.models.generate_content(
        model='gemini-3.1-flash-lite-preview',
        contents=system_instruction + "\n\n" + user_prompt,
        config={
            'temperature': 0.15,           # Low enough for RAG facts, high enough for natural speech
            'top_p': 0.85,                 # Cut off the "long tail" of weird hallucinated vocabulary
            'top_k': 40,                   # Only sample from the top 40 most likely next words
            'max_output_tokens': 800,      # Prevent the avatar from rambling for 3 minutes, but account for tokens required for thought
            'stop_sequences': ["User:", "Client:"] # Stop the LLM if it tries to roleplay the user
        }
    )
    
    raw_text = response.text
    
    speech_match = re.search(r"<speech>(.*?)</speech>", raw_text, re.DOTALL | re.IGNORECASE)
    thought_match = re.search(r"<thought>(.*?)</thought>", raw_text, re.DOTALL | re.IGNORECASE)
    
    if speech_match:
        final_spoken_answer = speech_match.group(1).strip()
    else:
        # Fallback just in case the LLM fails to use the tags
        final_spoken_answer = raw_text

    if thought_match:
        thought = thought_match.group(1).strip()
    else:
        thought = 'Invalid thoughts'
        
    return final_spoken_answer, thought


In [ ]:
prompt(generate_rag_response_v3, get_closest_matches)

Queries have been loaded.
Query: What kind of agent should I use for emails?
Docs: [{'id': 'req_044', 'document': 'Implement an email automation system to send alerts to traders when specific stock price thresholds are met or breached.', 'integration_level': 'low', 'domain': 'Financial Tech & Trading', 'avatar_response': 'Understood. We can implement a basic script or automation tool for this.'}, {'id': 'req_064', 'document': 'Create a spreadsheet automation to categorize incoming customer support emails based on keywords like "refund," "shipping," or "product inquiry."', 'integration_level': 'low', 'domain': 'Retail & E-commerce', 'avatar_response': 'A simple scripting solution can be developed to automate the classification and organization of your support emails.'}, {'id': 'req_063', 'document': 'Set up a rule-based system to automatically send welcome emails to new subscribers and discount codes to customers who abandon their shopping carts.', 'integration_level': 'low', 'domain': 

### Generator Fine-Tuning
Generator Fine-Tuning is usually done for 2 apparent reasons:
1. **Cost & Latency Reduction**  
If I fine-tune a model to always speak like my AI Agency Avatar, I can entirely delete the huge system prompt and Few-Shot examples, allowing for a very short prompt which saves massive amounts of API tokens and makes the response much faster. 
2. **Hyper-specific Formatting**  
If I needed the LLM to output a complex json schema that would be too hard to describe via prompt engineering

Honestly i dont need to do Generator Fine-Tuning. For a prototype application, prompt engineering is accurate enough and far more cost-efficient as fine-tuning an LLM like gemini-3.1-flash-lite that likely has a massive number of tokens compared to MiniLM. I would need to manually write or generate and craft a dataset of at least 100-500 perfect examples of [Context + Question] mapped to [Avatar Speech], which is very time-consuming. I think my prompt engineering and decoding tuning are already producig great results, so fine-tuning will yield very little noticeable improvement. Later on, if this avatar moves on to enterprise development, perhaps this would be more useful to reduce the cost of API tokens.

## Input Enhancement
I found several types of input enhancement techniques for RAG architecture: HyDE (Hypothetical Document Embeddings) and Query Decomposition (TOC). The issue is that since the final product would be an audio avatar, there is a requirement to balance accuracy and latency. Each input enhancement that I add is forcing the system to make an entirely separate LLM API call before it even searches the database. The human client talking to the avatar will likely get impatient waiting for the avatar to make a response.
1. **HyDE / Query2Doc:**  
    - *How it works*: When a client asks something like "How much for a defect tracking app?", I ask the LLM to write a fake hypothetical document answering the question, then vectorise that entire fake document and search the ChromaDB database with it. Since I am comparing a documet to a document (instead of a query to document comparison), the semantic match would be much stronger.
    - *Am i gna do this?*: No. I already fine-tuned the MiniLM model, which achieved a mathematically perfect 100% Hit Rate @ 3 on my evaluatin dataset. HyDE is used when retrievers are struggling to find documents. Maybe later on when I get the actual database, I might need to use this if the retriever struggles.

2. **TOC / Query Decomposition**:  
    - *How it works*: When a client asks a compound question like "I need to automate company emails and also build a robot arm for my factory" then an LLM splits this into two sub-queries: "Automate emails" and "Build factory robot arm". Then the retriever would search the ChromaDB twice separately then combine the documents.
    - *Am i gna do this?*: This would be useful if clients frequently ask multi-part questions. This can be explored later on, but for now I'm just going to skip it to save latency and assume clients will ask one thing at a time.

What I will implement is Conversational Rewriting. Since the end-user is talking to an avatar, it is likely that they will be having a conversation rather than a one-shot Q&A. In conversations, people usually use pronouns and reference the conversation history more often than not.  
For example,  
User: I want to automate my emails.    
Avatar: (biblically flawless perfect answer)  
User: How much would *that* cost?  
  
If this passes into the vector database, it will fail because the database has no cluewhat "that" is.  
To alleviate this issue, I need a highly optimised and super fast Query Rewriting Step that takes the conversational history and rewrites the query so that the vector database can understand it. 

### Conversational Rewriting

In [ ]:
def rewrite_query(chat_history, latest_user_query):
    # should be super fast to save latency, so no complex prompting
    rewrite_instruction = """
    You are a query rewriter. Look at the chat history and the latest user query.
    If the user's query contains pronouns (it, that, they, etc.) or relies or previous context, rewrite it into a single standalone sentence.
    If the query is already standalone, just output the exact same query.
    DO NOT answer the question. ONLY output the rewritten query.
    """

    prompt = f"""
    Chat History:
    {chat_history}

    Latest Query: {latest_user_query}

    Rewritten Query:
    """

    llm_client = get_llm_client()
    response = llm_client.models.generate_content(
        model="gemini-3.1-flash-lite-preview", 
        contents=rewrite_instruction + '\n' + prompt,
        config={'temperature': 0.0} # 0 temperature for robotic precision
    )

    return response.text.strip()

In [ ]:
llm_client = get_llm_client()
for m in llm_client.models.list():
    print(m.name if m.supported_actions==['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent'] else None)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite








models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite

models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview


























In [ ]:
def pipeline(prompts: list[str], generator: Callable[..., Any]=generate_rag_response_v3, retriever: Callable[..., Any]=get_closest_matches):
    chat_hist = ''
    for prompt in prompts:
        new_prompt = rewrite_query(chat_hist, prompt)
        retrieved = retriever(collection_minilm_finetuned, finetuned_qemb(new_prompt), k=3)
        final_answer, thoughts = generator(user_query=new_prompt, retrieved_documents=retrieved)

        result = f"""
Query: {new_prompt}
Docs: {retrieved}
Thinking: {thoughts}
Response: {final_answer}\n
        """
        print(result)
        chat_hist += result

In [ ]:
prompts = ["I need a system to scan my construction site to make periodical reports.", "What sort of technology will this use?", "How do I implement it?"]
pipeline(prompts)


Query: I need a system to scan my construction site to make periodical reports.
Docs: [{'id': 'req_092', 'document': 'Create a real-time analytics dashboard that monitors construction site progress, comparing drone imagery with BIM models to identify discrepancies and track milestones.', 'integration_level': 'mid', 'domain': 'Manufacturing & Construction', 'avatar_response': 'This is well within our expertise. Our mid-level integration solutions offer robust AI capabilities for real-time analysis and enhanced operational efficiency.'}, {'id': 'req_086', 'document': 'Build a simple optical character recognition (OCR) tool to convert handwritten construction site safety inspection checklists into structured digital spreadsheet data.', 'integration_level': 'low', 'domain': 'Manufacturing & Construction', 'avatar_response': 'We can certainly assist with that. Our low-level integration services are perfect for automating routine tasks and improving data flow.'}, {'id': 'req_061', 'document

It tentatively works, but the conversation flow feels robotic and the chatbot is like a broken record, repeating the exact same jargon over and over again. The two main causes are likely 
1. that the generator does not have access to the chat history, so every question is essentially the first time it has ever spoken to the user, so it greets the same way every time
2. that my prompt is overly strict; my prompt says "You MUST use the provided suggested avatar response". Since i set the temp to low (0.15) to prevent hallucinations, the LLM is taking the instruction literally. It sees that document 2 has the suggest response and forces that exact string of text into every answer.  
  
Let me fix this

In [ ]:
def generate_rag_response_v4(user_query, retrieved_documents, chat_history=""):
    # 1. Format the context
    context_str = "\n\n".join([
        f"[Doc {i+1}]:\nDocument: {doc['document']}\nIntegration Level: {doc['integration_level']}\nDomain: {doc['domain']}\nSuggested Tone/Response: {doc['avatar_response']}" 
        for i, doc in enumerate(retrieved_documents)
    ])
    
    # 2. Build the engineered prompt
    system_instruction = """
    You are the voice of a digital avatar representing an expert AI integration agency. 
    Your job is to assess client requests and respond naturally in conversation.
    
    You will be provided with 'Suggested Tone/Response' and 'Integration Level'. Use these to guide the VIBE and direction of your answer, but DO NOT just copy-paste them repeatedly.
    Vary your phrasing naturally like a real human having a continuous conversation.
    
    The <context> block contains all the relevant documents your agency possesses regarding the <question>.
    Keep your response professional, warm, and suitable for text-to-speech audio generation.
    
    CRITICAL RULES:
    1. You must ONLY supply facts derived from the <context> given. If there is no info, say: "Sorry, I do not have enough information to provide an adequate answer."
    2. Read the <chat_history>. Do NOT repeat greetings or exact phrases you have already said recently. Make your response flow logically from the previous turn.
    3. Always cite your sources safely by saying things like "As mentioned in our capabilities..." rather than reading raw bracket citations out loud.
    
    FORMATTING:
    First write your logic inside <thought> tags. Then, write the final spoken response inside <speech> tags.
    
    <example>
    <thought>
    The user is asking about automating email sorting. Document 1 talks about categorizing customer support emails based on keywords. Integration level is low, domain is retail. The suggested avatar response is "A simple scripting solution can be developed." I will synthesize this into a warm, confident response.
    </thought>
    <speech>
    Yes, we can absolutely help with your emails! A simple scripting solution can be developed to automatically categorize your incoming support emails based on keywords like 'refund' or 'shipping'. Because this is a low-level integration, we can set this up for you very quickly.
    </speech>
    </example>
    """
    
    user_prompt = f"""
    <chat_history>
    {chat_history}
    </chat_history>
    
    <context>
    {context_str}
    </context>
    
    <question>
    {user_query}
    </question>
    """
    
    llm_client = get_llm_client()
    
    response = llm_client.models.generate_content(
        model='gemini-3.1-flash-lite-preview',
        contents=system_instruction + "\n\n" + user_prompt,
        config={
            'temperature': 0.3,            # Raised slightly so the LLM can vary its phrasing!
            'top_p': 0.85,                 
            'top_k': 40,                   
            'max_output_tokens': 800,      
            'stop_sequences': ["User:", "Client:"] 
        }
    )
    
    raw_text = response.text
    
    import re
    speech_match = re.search(r"<speech>(.*?)</speech>", raw_text, re.DOTALL | re.IGNORECASE)
    thought_match = re.search(r"<thought>(.*?)</thought>", raw_text, re.DOTALL | re.IGNORECASE)
    
    if speech_match:
        final_spoken_answer = speech_match.group(1).strip()
    else:
        final_spoken_answer = raw_text
    if thought_match:
        thought = thought_match.group(1).strip()
    else:
        thought = 'Invalid thoughts'
        
    return final_spoken_answer, thought

def pipeline_v2(prompts: list[str], generator=generate_rag_response_v4, retriever=get_closest_matches):
    # We only want to store the actual spoken conversation for the history, not the debug text
    clean_history = "" 
    
    for prompt in prompts:
        # Step 1: Rewrite the query based on history
        new_prompt = rewrite_query(clean_history, prompt)
        
        # Step 2: Retrieve documents
        retrieved = retriever(collection_minilm_finetuned, finetuned_qemb(new_prompt), k=3)
        
        # Step 3: Generate response (Pass the clean_history to the generator!)
        final_answer, thoughts = generator(user_query=new_prompt, retrieved_documents=retrieved, chat_history=clean_history)
        # Output the debug info for the notebook
        result_debug = f"""
Query: {prompt}
Rewritten Query: {new_prompt}
Docs: {retrieved}
Thinking: {thoughts}
Response: {final_answer}\n
        """
        print(result_debug)
        print("-" * 50)
        
        # Step 4: Add the turn to the clean conversation history
        clean_history += f"User: {prompt}\nAvatar: {final_answer}\n"


In [ ]:
pipeline_v2(prompts)


Query: I need a system to scan my construction site to make periodical reports.
Rewritten Query: I need a system to scan my construction site to generate periodical reports.
Docs: [{'id': 'req_092', 'document': 'Create a real-time analytics dashboard that monitors construction site progress, comparing drone imagery with BIM models to identify discrepancies and track milestones.', 'integration_level': 'mid', 'domain': 'Manufacturing & Construction', 'avatar_response': 'This is well within our expertise. Our mid-level integration solutions offer robust AI capabilities for real-time analysis and enhanced operational efficiency.'}, {'id': 'req_061', 'document': 'Automate the daily extraction of product review data from third-party websites and consolidate it into a weekly report.', 'integration_level': 'low', 'domain': 'Retail & E-commerce', 'avatar_response': 'Understood. We can implement a targeted web scraping script for that specific data extraction and reporting.'}, {'id': 'req_086',

Works great now.  
The generator now reads the history as well, and I changed the "MUST" to "guide the VIBE", and slightly higher temperature (0.3) to give the LLM a bit more freedom to use different vocab while still staying factually strict.

### Query Decomposition
to be implemented when the need arises

## RAG Pipeline Enhancements
This refers to optimising the overall process of RAG to achieve better performance results. Mainly consists of Adaptive Retrieval and Iterative RAG.
1. **Iterative RAG (ITER-RETGEN)**  
    - *How it works*: The LLM generates a draft answer, realises it's missing info, executes a new search query, retrieves more docs, and generates a final improved answer.
    - *Am i gna do this?*: no. firstly, iterative RAG seems really complicated and it's designed for AI researchers or legal assistants answering complex multi-hop questions like "Cross Reference to 2018 tax law with the 2020 amendment". For my avatar, iterative rag is massive overkill. It aslo forces multiple sequential LLM calls, meaning the avatar could sit in silence 8-10 seconds before speaking, which causes some QOL issues for the end-user.
2. **Adaptive Retrieval (Semantic Routing)**  
    - *How it works*: An algorithm of sorts that determines if I even need to search the database at all.
    - Rule-based vs Model-based
        - Rule-based (e.g. regex matching for keywords like "hello") is too brittle
        - Model-based (using a fast LLM or classifier) is way better
    - Examples of model-based adaptive retrieval:
        - Self-RAG uses a trained generator to determine whether to perform a retrieval based on the retrieve token under different user queries
        - Ren et al. used "Judgement Prompting" to determine whether LLMs can answer relevant questions and whether their answers are correct or not, thereby assisting in determining the necessity of a retrieval
        - SKR uses the ability of LLMs themselves to judge in advance whether they can answer the question, and if they can answer the question, no retrieval is performed
        - Rowen translates a question into multiple languages and checks for answer consistency across these languages, using the results to determine the need for retrieval
        - AdaptiveRAG dynamically decides whether to retrieve based on the query complexity by a classifier, which is a smaller LM
    - Am i gonna do this: yes. But just model-based, and probably gonna most closely align with AdaptiveRAG and SKR

Adaptive Retrieval is pretty important because if the user says something like "Hello how are you" or "Okay, thanks for the info", my current pipeline basically frantically vectorises those strings, searches ChromaDB, pulls up random documents, tries to answer it like it's a legitimate question. I need an adaptive router that looks at the query and decides if it requires searching the database or if it should just chat.

### Adaptive Retrieval (Model-Based)

In [ ]:
def adaptive_router(chat_history, latest_user_query):
    """
    Decides whether the query requires retrieving documents from the database (RAG)
    or if it's just casual conversation/small talk (CHAT).
    """
    router_instruction = """
    You are a routing system for an AI Agency digital avatar. 
    Look at the user's latest query. 
    - If the user is asking about services, capabilities, costs, tech, or asking for help building something, output EXACTLY the word: RAG
    - If the user is just saying hello, goodbye, expressing thanks, or making casual small talk, output EXACTLY the word: CHAT
    
    Output nothing else. Only RAG or CHAT.
    """
    
    llm_client = get_llm_client()
    response = llm_client.models.generate_content(
        model="gemini-2.5-flash-lite", # Extremely fast and cheap
        contents=f"{router_instruction}\n\nQuery: {latest_user_query}",
        config={'temperature': 0.0} 
    )
    
    route = response.text.strip().upper()
    return route if route in ["RAG", "CHAT"] else "RAG" # Default to RAG if it glitches


def generate_chat_response(user_query, chat_history):
    system_instruction = """
    You are the voice of a digital avatar representing an expert AI integration agency.
    The user is making casual conversation (e.g., greetings, thanks, farewells).
    Respond warmly, professionally, and very concisely.
    """
    
    user_prompt = f"<chat_history>{chat_history}</chat_history>\n\n<question>{user_query}</question>"
    
    llm_client = get_llm_client()
    response = llm_client.models.generate_content(
        model='gemini-3.1-flash-lite-preview',
        contents=system_instruction + "\n\n" + user_prompt,
        config={'temperature': 0.5, 'max_output_tokens': 100} # Higher temp for natural chat
    )
    
    return response.text.strip()

def pipeline_v3(prompts: list[str], generator=generate_rag_response_v4, retriever=get_closest_matches, generator_lite=generate_chat_response):
    clean_history = "" 
    
    for prompt in prompts:
        print(f"User: {prompt}")
        
        # --- 1. ADAPTIVE RETRIEVAL (ROUTING) ---
        route = adaptive_router(clean_history, prompt)
        print(f"[System: Routed to {route}]")
        
        if route == "CHAT":
            final_answer = generator_lite(prompt, clean_history)
            thoughts = "No thoughts needed for casual chat."
            
        else: # route == "RAG"
            # --- 2. INPUT ENHANCEMENT ---
            new_prompt = rewrite_query(clean_history, prompt)
            
            # --- 3. RETRIEVAL ---
            retrieved = retriever(collection_minilm_finetuned, finetuned_qemb(new_prompt), k=3)
            
            # --- 4. GENERATION ---
            final_answer, thoughts = generator(user_query=new_prompt, retrieved_documents=retrieved, chat_history=clean_history)

        print(f"Thinking: {thoughts}")
        print(f"Avatar: {final_answer}\n")
        print("-" * 50)
        
        clean_history += f"User: {prompt}\nAvatar: {final_answer}\n"


In [ ]:
mixed_prompts = [
    "Hello there! How are you doing today?", 
    "I need an app to scan construction sites.", 
    "How much does it cost?", 
    "Okay, thank you so much for your help!"
]

pipeline_v3(mixed_prompts)

User: Hello there! How are you doing today?
[System: Routed to CHAT]
Thinking: No thoughts needed for casual chat.
Avatar: Hello! I'm doing great, thank you for asking. How can I assist you with your AI integration needs today?

--------------------------------------------------
User: I need an app to scan construction sites.
[System: Routed to RAG]
Thinking: The user is asking for an app to scan construction sites. Looking at the context, there are two relevant options: Doc 1 (real-time analytics using drone imagery and BIM models) and Doc 2 (OCR for digitizing handwritten safety checklists). Since the user's request is broad, I should present both options to see which aligns better with their specific needs, maintaining a professional and helpful tone.
Avatar: That sounds like a great way to modernize your operations. Depending on your specific goals, we have a couple of ways to approach this. 

If you are looking to digitize your paperwork, we can build a tool that uses optical char

Looking great! However, the model is asking a lot of questions. That's fine though.

## Conclusion

As of 2/4/26, the improved RAG systems seems adequate to be integrated into the app.  
### Features Added
1. Data Evaluation & Benchmarking  
  
    - Synthetic Evaluation Dataset: Generated 300 realistic, distinct client queries mapped directly to correct "gold standard" document IDs to create an objective baseline
    - Retriever Benchmarking: Compared bi-encoder DPR against all-MiniLM-L6-v2 dense vector model using P@10, proving MiniLM was superior for this specific agency domain
  
2. Retriever Enhancements
  
    - Retriever Fine-Tuning: Successfully trained the base MiniLM model on my specfic domain data using MNRL. The fine-tuned model achieved a mathematically perfect 100% Hit Rate @ 3, ensuring the correct documents are always retrieved.
  
3. Generator Enhancements
  
    - Advanced Prompt Engineering:
        - Context Delimiters: Structured the retrieved data meticulously to pass Domain, Integration Level, and Document content directly to the LLM.
        - Anti-Hallucination Bounds: Implemented strict instructions forcing the LLM to say "I don't have enough information" if the context doesn't hold the answer.
        - Hidden Chain of Thought (CoT): Forced the LLM to verify its own logic inside <thought> tags before generating the spoken response inside <speech> tags.
        - Few-Shot Examples: Guided the avatar's specific "Vibe" and tone using a perfect example interaction within the system instruction.
    - Decoding Tuning: Tuned generation hyperparameters (temperature=0.3, top_p=0.85, top_k=40, max_output_tokens=800, stop_sequences) to ensure the avatar speaks consistently without rambling or hallucinating sides of the conversation.
  
4. Input & Pipeline Enhancements
  
    - Conversational Query Rewriting: Implemented a lightning-fast pre-processing step using gemini-2.5-flash-lite to resolve pronouns (e.g., rewriting "How much does that cost?" to "How much does the defect tracking app cost?") ensuring the vector search never breaks during conversation flow.
    - Generator Memory Integration: Wired the clean_history string directly into the generator prompt so the avatar is aware of what it just said, preventing unnatural, robotic repetition of greetings or corporate jargon.
    - Adaptive Retrieval (Semantic Routing): Built a high-speed router to classify the intent of a user's prompt as RAG or CHAT. This allows the system to bypass ChromaDB entirely for small talk, saving massive amounts of audio latency while avoiding database pollution.

### Future Enhancements (For Enterprise Scaling)
While the current pipeline is perfectly optimised for a high-performance prototype with low audio latency, there are architectural upgrades needed if this is deployed at enterprise scale.
  
1. Retriever Scaling
  
    - Hybrid Retrieval (Vector + Keyword): In the future, dense vectors might miss highly specific acronyms (e.g., "ERP-7v2"). Combining MiniLM vector search with a sparse lexical search algorithm (like BM25) provides the ultimate safety net.
    - Cross-Encoder Reranking: When the document database grows from 100 to 10,000 documents, you will pull the Top 20 documents. A Cross-Encoder model is then run locally to deeply analyze and re-sort those 20 documents, passing only the absolute best 3 to the LLM.
  
2. Input & Query Scaling
  
    - HyDE (Hypothetical Document Embeddings): If user queries remain too short or vague, an LLM can generate a "fake/hypothetical" answer first. Embedding that hypothetical answer yields vastly deeper semantic search results.
    - Query Decomposition (TOC): If users begin requesting massive, compound architectures (e.g., "I need an email bot, an HR onboarding system, and a factory robotic arm"), an LLM breaks the query into three distinct RAG searches and synthesizes a master response.
  
3. Generator Optimisation
  
    - Iterative RAG (ITER-RETGEN): Allowing the LLM to evaluate its own draft answer, realize it has a knowledge gap, and dynamically trigger a second search of the database before talking.
    - Generator Fine-Tuning: If API token cost becomes an issue, the Gemini model can be trained on the specific AI Agency Avatar responses. This allows us to delete the entire system prompt, drastically reducing token usage and latency.

### References
Zhao, P., Zhang, H., Yu, Q., Wang, Z., Geng, Y., Fu, F., Yang, L., Zhang, W., Jiang, J., & Cui, B. (2024, February 29). _Retrieval-Augmented Generation for AI-Generated Content: a survey_. arXiv.org. https://arxiv.org/abs/2402.19473

# Fine Tuning a new model to fit the new database

In [23]:
# Load the Sales Tutor Eval Dataset
with open("../data/eval_qa_dataset.json", "r") as f:
    sales_eval_dataset = json.load(f)

# Format the Data into InputExamples for SentenceTransformers
sales_train_examples = []
for item in sales_eval_dataset:
    # We want the retriever to find the rubric regardless of whether the rep's answer was good or bad
    query = item["rep_answer_query"]
    document_text = item["scenario_context"]
    sales_train_examples.append(InputExample(texts=[query, document_text]))

# Create a DataLoader
sales_train_dataloader = DataLoader(sales_train_examples, shuffle=True, batch_size=8)

# Load base model
sales_minilm = SentenceTransformer("all-MiniLM-L6-v2")

# Loss function (MultipleNegativesRankingLoss uses in-batch negatives)
sales_train_loss = losses.MultipleNegativesRankingLoss(model=sales_minilm)

print("Starting Fine-tuning for Sales Tutor Retriever")
from sentence_transformers import fit_mixin
sales_minilm.fit(
    train_objectives=[(sales_train_dataloader, sales_train_loss)],
    epochs=4,
    warmup_steps=int(len(sales_train_dataloader) * 0.1),
    show_progress_bar=True
)

sales_minilm.save("../models/finetuned-minilm-sales-tutor")
print("Model saved successfully to ../models/finetuned-minilm-sales-tutor")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 17763.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting Fine-tuning for Sales Tutor Retriever


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 13.55it/s]

Model saved successfully to ../models/finetuned-minilm-sales-tutor


In [24]:
# 1) Re-load the fine-tuned Sales Tutor model we just built
sales_model_path = "../models/finetuned-minilm-sales-tutor"
finetuned_sales_model = SentenceTransformer(sales_model_path)

def sales_qemb(q):
    return finetuned_sales_model.encode(q, normalize_embeddings=True).tolist()

# 2) Encode the scenario documents to inject into a fresh evaluation collection
print("Embedding Sales Scenarios with newly finetuned model...")
with open("../data/sales_scenarios.json", "r") as f:
    sales_data = json.load(f)
    
sales_docs = [item["document"] for item in sales_data]
sales_metas = [item["metadata"] for item in sales_data]
sales_ids = [item["id"] for item in sales_data]

sales_doc_emb = finetuned_sales_model.encode(sales_docs, normalize_embeddings=True)

# 3) Upsert into local DB
# client is already initialized earlier as get_db_client() with PersistentClient and DB_PATH
try:
    client.delete_collection("eval_collection_sales")
except Exception:
    pass

collection_sales = client.create_collection(name="eval_collection_sales")
collection_sales.upsert(
    documents=sales_docs,
    metadatas=sales_metas,
    ids=sales_ids,
    embeddings=sales_doc_emb.tolist()
)
print("Successfully injected Fine-Tuned encodings for Sales Scenarios.")

# 4) Prep Evaluation queries & gold lists from the eval_qa_dataset
with open("../data/eval_qa_dataset.json", "r") as f:
    eval_qa = json.load(f)
    
eval_sales_queries = [item["rep_answer_query"] for item in eval_qa]
eval_sales_gold = [[item["scenario_id"]] for item in eval_qa]

# 5) Run evaluations at K=1
sales_hit_rate_at_1 = eval_retriever(collection_sales, sales_qemb, eval_sales_queries, eval_sales_gold, k=1)
print(f"Fine-Tuned Sales Model Hit Rate @ 1: {sales_hit_rate_at_1}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14813.24it/s]


Embedding Sales Scenarios with newly finetuned model...
Successfully injected Fine-Tuned encodings for Sales Scenarios.
Fine-Tuned Sales Model Hit Rate @ 1: 1.0
